<a href="https://colab.research.google.com/github/Shadoww002/NLP/blob/main/Phase-1/Phase_1_The_Linguistic_Foundation_(Classical_NLP).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



**Task 1.1: The Art of Tokenization**




In [1]:
import re

In [2]:
text = "Hello! I'm learning NLP in 2026. Isn't it amazing? @Shadoww002"

In [3]:
text = text.lower()

In [4]:
clean_text = re.sub(r'[^a-z0-9\s]',"",text)

In [5]:
tokens = clean_text.split()

In [6]:
tokens

['hello',
 'im',
 'learning',
 'nlp',
 'in',
 '2026',
 'isnt',
 'it',
 'amazing',
 'shadoww002']

**Task 1.2: Normalization (Stemming vs. Lemmatization)**

In [7]:
import nltk
from nltk.stem import PorterStemmer , WordNetLemmatizer
nltk.download("wordnet")

[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [8]:
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer

In [10]:
words = ["running" , "flies","fleet" , "caring"]

In [11]:
print("Stemming results:")
x = []
for w in words:
  x.append(stemmer.stem(w))
print(x)

Stemming results:
['run', 'fli', 'fleet', 'care']


In [20]:
print("\nLemmatization results:")
print([lemmatizer.lemmatize(w, pos='v') for w in words]) # pos='v' tells it they are verbs


Lemmatization results:
['run', 'fly', 'feet', 'care']


**Task 1.3: Stop Words & Vocabulary Building**

In [21]:
import torch

# Our raw "dataset"
data = [
    "The hacker is attacking the firewall",
    "Firewall blocked the malicious virus",
    "Virus is spreading fast"
]

# 1. Simple Tokenization & Lowercase
tokenized_data = [sentence.lower().split() for sentence in data]

In [22]:
tokenized_data

[['the', 'hacker', 'is', 'attacking', 'the', 'firewall'],
 ['firewall', 'blocked', 'the', 'malicious', 'virus'],
 ['virus', 'is', 'spreading', 'fast']]

In [23]:
# 2. Create the Vocabulary (Unique words only)
# We add <PAD> at index 0 and <UNK> at index 1
vocab = {"<PAD>": 0, "<UNK>": 1}
for sentence in tokenized_data:
    for word in sentence:
        if word not in vocab:
            vocab[word] = len(vocab)

print(f"Vocabulary: {vocab}")

Vocabulary: {'<PAD>': 0, '<UNK>': 1, 'the': 2, 'hacker': 3, 'is': 4, 'attacking': 5, 'firewall': 6, 'blocked': 7, 'malicious': 8, 'virus': 9, 'spreading': 10, 'fast': 11}


In [24]:
# 3. Numericalize: Convert words to their ID
numerical_data = [[vocab.get(word, 1) for word in sentence] for sentence in tokenized_data]

print(f"\nNumerical Sequences: {numerical_data}")


Numerical Sequences: [[2, 3, 4, 5, 2, 6], [6, 7, 2, 8, 9], [9, 4, 10, 11]]


**Task 1.3 (Part B): N-grams & Language ModelsA Language Model is simply something that assigns a probability to a sequence of words.**

**N-grams are the simplest way to do this. We look at chunks of $n$ words to predict the next one.**

Unigram ($n=1$): Every word is independent.

Bigram ($n=2$): The probability of a word depends only on the previous word.
Example: If the current word is "San", the probability of the next word being "Francisco" is very high.

Trigram ($n=3$): Depends on the previous two words.The Math: To find the probability of a word $w$ given a previous word $h$ (history), we just count how many times they appeared together in our training data:$$P(w|h) = \frac{count(h, w)}{count(h)}$$


**Task 1.3 (Part C): Edit Distance (The Core of Autocorrect)**

If a user types "hasker" instead of "hacker", how does the computer know what they meant? We use Levenshtein Edit Distance.

It counts the minimum number of operations required to change one word into another. The operations are:

Insertion: adding a letter.

Deletion: removing a letter.

Substitution: swapping one letter for another.

**Milestone 1: The "Auto-Correct" System**

In [32]:
import re
from collections import Counter

# 1. Our 'Mini-Corpus' (In reality, this would be millions of words)
text_data = "the be to of and a in that have i it for not on with he as you do at this but his by from they we say her she or an will my one all would there their what so up out if about who get which go me when make can like time no just him know take people into year your good some could them see other than then now look only come its over think also back after use two how our work first well even new want because any these give day most us"
words = re.findall(r'\w+', text_data.lower())
VOCAB_COUNTS = Counter(words)
TOTAL_WORDS = sum(VOCAB_COUNTS.values())

def get_probability(word):
    # P(w) = count of word / total words
    return VOCAB_COUNTS[word] / TOTAL_WORDS

def generate_edit_1(word):
    """Generates all words 1 edit away from the input"""
    letters    = 'abcdefghijklmnopqrstuvwxyz'
    splits     = [(word[:i], word[i:])    for i in range(len(word) + 1)]
    deletes    = [L + R[1:]               for L, R in splits if R]
    substitutes = [L + c + R[1:]           for L, R in splits if R for c in letters]
    insertions = [L + c + R                for L, R in splits for c in letters]
    return set(deletes + substitutes + insertions)

def autocorrect(word):
    # 1. If word is already in vocab, it's correct!
    if word in VOCAB_COUNTS:
        return word

    # 2. Generate candidates 1 edit away
    candidates = generate_edit_1(word)

    # 3. Filter candidates to only those that actually exist in our vocab
    valid_candidates = [w for w in candidates if w in VOCAB_COUNTS]

    if not valid_candidates:
        return "No suggestion found"

    # 4. Use Probability to pick the best one
    # This is a 'Unigram' approach
    best_guess = max(valid_candidates, key=get_probability)
    return best_guess

In [37]:
# Testing the system
print(f"Correction for 'har': {autocorrect('har')}")

Correction for 'har': her


In [39]:
# Testing the system
print(f"Correction for 'et': {autocorrect('et')}")

Correction for 'et': at


In [41]:
# Testing the system
print(f"Correction for 'thear': {autocorrect('thear')}")

Correction for 'thear': their


In [42]:
# Testing the system
print(f"Correction for 'hii': {autocorrect('hii')}")

Correction for 'hii': his
